# AI Engineering Study Agent — System Design Walkthrough

This notebook is a self-teaching guide. It walks through every layer of the system — RAG, agentic orchestration, prompt engineering, streaming, auth, persistence, and CI/CD — using actual code from this repo.

**Who this is for:** someone who understands basic Python and web concepts but has not built a production AI application before.

**How to read it:** read the markdown cells first, then look at the code cells. The code is copied directly from the live codebase — it is not simplified. That is intentional.

---

## Table of Contents

1. [System Map — What We're Building](#1-system-map)
2. [RAG Part 1 — Ingestion (One-Time Pipeline)](#2-rag-ingestion)
3. [RAG Part 2 — Retrieval at Query Time](#3-rag-retrieval)
4. [Agentic Orchestration — The 4-Phase Pipeline](#4-agent-pipeline)
5. [Prompt Engineering — Every System Prompt Explained](#5-prompt-engineering)
6. [Streaming — How Tokens Reach the Browser](#6-streaming-sse)
7. [Authentication — Supabase OTP + JWT](#7-authentication)
8. [Persistence — Threads, Messages, and Graphs](#8-persistence)
9. [CI/CD — From Git Push to Live Production](#9-cicd)
10. [Infrastructure — Cloud Run, Vercel, and Cold Starts](#10-infrastructure)
11. [Key Patterns Worth Remembering](#11-key-patterns)

---
## 1. System Map — What We're Building <a id='1-system-map'></a>

Before diving into code, picture the full system.

```
User's browser
    │
    │  HTTPS
    ▼
Vercel (React frontend)              ◄─── Vite + TypeScript
    │
    │  POST /api/chat  (SSE stream)
    ▼
Cloud Run (FastAPI backend)          ◄─── Python, uvicorn
    │
    ├── Phase 0: Router → which path to take?
    ├── Phase 1a: RAG worker → search the book
    │           Research worker → optional web search
    ├── Phase 1b: Graph worker → build/update knowledge graph
    ├── Phase 2: Synthesise → write the answer, stream tokens
    └── Phase 3: Node enrichment (background, after response)
    │
    ├── FAISS index (loaded in RAM at startup)  ◄─── book embeddings
    └── Supabase Postgres  ◄─── threads, messages, graphs
```

### The three categories of complexity

- **RAG** — the book is the knowledge source. It cannot be sent to the LLM in full (too large). Instead we embed every passage, store the vectors, and at query time fetch only the relevant ones.
- **Agent pipeline** — instead of one big LLM call, the system runs specialised workers in a coordinated sequence. Each worker does one job well.
- **Infrastructure** — the backend sleeps when idle (Cloud Run `min-instances=0`) to keep costs near zero. The frontend has a `Prepare` button that wakes it before the first message.

---
## 2. RAG Part 1 — Ingestion (One-Time Pipeline) <a id='2-rag-ingestion'></a>

### What problem does RAG solve?

The book has ~400 pages. An LLM has a context window — a cap on how many tokens it can receive in one call. Even if we could fit the whole book in the context, it would be expensive and slow.

**RAG** (Retrieval-Augmented Generation) is a pattern where:
1. You embed the book offline (one-time work)
2. At query time, you search for relevant passages
3. You send only those passages to the LLM as context

Think of it like a librarian who reads every book once, builds an index, and then quickly finds the right page when you ask a question — instead of re-reading the entire library every time.

### The ingestion pipeline

```
PDF file
  │
  ▼  (Step 1: chunk)
Parent chunks (2048 chars)   ← these are returned to the user
Child chunks  (512 chars)    ← these are embedded + searched
  │
  ▼  (Step 2: embed)
384-dimensional float vectors (one per child chunk)
  │
  ▼  (Step 3: save)
FAISS index (index.faiss)    ← fast nearest-neighbour search
Parent docs  (parent_docs.pkl) ← full sections, loaded separately
```

### Why two chunk sizes (parent + child)?

This is called **parent-document retrieval**.

- **Small chunks** are better for *finding* the right passage because the embedding captures a tighter semantic signal.
- **Larger chunks** are better for *reading* because they include more context around the match.

So: embed the small chunk, search with the small chunk, but return the parent (large) chunk to the LLM.

In [ ]:
# ingestion/ingest.py — the full ingestion pipeline
# This script runs once. Its output (index.faiss, parent_docs.pkl) is
# packaged into the Docker image and loaded at startup.

from langchain_community.vectorstores import FAISS
from chunker import chunk_pdf
from config import BOOKS, OUTPUT_DIR
from sentence_embedder import SentenceEmbedder

def run(pdf_path, book_key):
    book_meta = BOOKS[book_key]

    # Step 1: chunk ─────────────────────────────────────────────────────────
    # chunk_pdf returns (parent_doc, [child_doc, ...]) pairs
    pairs = chunk_pdf(pdf_path, book_key=book_key, book_author=book_meta["author"])

    parent_docs = [parent for parent, _ in pairs]
    child_docs  = [child for _, children in pairs for child in children]
    # parent_docs: ~200-300 chunks, each 2048 chars (full sections)
    # child_docs:  ~800-1200 chunks, each 512 chars (search targets)

    # Step 2: embed + build FAISS ─────────────────────────────────────────────
    embedder = SentenceEmbedder()   # wraps sentence-transformers/all-MiniLM-L6-v2

    child_texts     = [doc.page_content for doc in child_docs]
    child_metadatas = [doc.metadata for doc in child_docs]
    # Each child_metadata has: chapter, page_number, section, book_title, book_author
    # This metadata rides alongside the vector — retrieved together when we search.

    vectorstore = FAISS.from_texts(
        texts=child_texts,
        embedding=embedder,
        metadatas=child_metadatas,
    )
    # FAISS.from_texts calls embedder.embed_documents(child_texts)
    # which returns a list of 384-float vectors, one per chunk.
    # FAISS stores them in a flat index — no approximate search, exact cosine similarity.

    # Step 3: save ─────────────────────────────────────────────────────────────
    vectorstore.save_local(str(OUTPUT_DIR))
    # Produces: data/faiss/index.faiss  (the vector index)
    #           data/faiss/index.pkl    (the child docstore: vectors → text)

    import pickle
    with open(OUTPUT_DIR / "parent_docs.pkl", "wb") as f:
        pickle.dump(parent_docs, f)
    # parent_docs.pkl: the full sections — loaded separately at query time

### How the PDF is chunked (font-size detection)

A naive approach would split text by paragraphs or fixed character counts. The problem: book PDFs don't have markdown headers — chapters and sections are just text rendered in a larger font.

The chunker uses `pdfplumber` which exposes every word's font size. By analysing font-size frequency, it auto-detects which sizes correspond to body text, section headers, and chapter titles.

In [ ]:
# ingestion/chunker.py — font-size-based section detection (simplified excerpt)

import pdfplumber
from collections import Counter

def _build_font_profile(pdf, sample_pages=30):
    """
    Auto-detect font sizes by sampling the first 30 pages.

    Key insight:
      - Body text is the MOST COMMON font size (by word count)
      - Headers are LARGER than body, appear less frequently
      - Chapter labels ("CHAPTER 3") have a distinct font on their own pages
    """
    size_counts = Counter()

    for page in pdf.pages[:sample_pages]:
        words = page.extract_words(extra_attrs=["fontname", "size"])
        for w in words:
            rounded = round(w["size"], 1)  # round to 1dp to group close sizes
            size_counts[rounded] += 1

    # Body text = most frequent size
    body_size = size_counts.most_common(1)[0][0]

    # Header candidates = sizes larger than body
    header_candidates = sorted(
        [s for s in size_counts if s > body_size + 1.0],
        reverse=True
    )

    return {
        "body":           body_size,
        "section_header": header_candidates[-1] if header_candidates else body_size + 2,
        "chapter_title":  header_candidates[0]  if header_candidates else body_size + 4,
    }

# Why not regex? Example: "Chapter 3" could appear as body text in a sentence.
# Font size is unambiguous — a chapter heading is ALWAYS rendered larger.
# This approach generalises across all O'Reilly-style PDFs.

### The embedding model

An **embedding** is a way to represent text as a vector of numbers. Two pieces of text that mean similar things will have vectors that point in similar directions — their **cosine similarity** will be high.

We use `sentence-transformers/all-MiniLM-L6-v2`:
- Produces **384-dimensional** vectors
- Runs on CPU (no GPU needed)
- ~80MB model size — small enough to bundle in Docker
- Fast enough: a batch of 500 chunks takes ~5 seconds on a laptop

In [ ]:
# ingestion/sentence_embedder.py — wraps sentence-transformers in a LangChain interface

from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings

class SentenceEmbedder(Embeddings):
    """
    LangChain-compatible wrapper for the sentence-transformers library.

    LangChain's FAISS integration expects an Embeddings object with two methods:
      - embed_documents(texts): list[str] → list[list[float]]
      - embed_query(text):      str       → list[float]
    """
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self._model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        # .encode returns a numpy array, .tolist() converts to plain Python lists
        return self._model.encode(texts, show_progress_bar=False).tolist()

    def embed_query(self, text):
        return self._model.encode([text], show_progress_bar=False)[0].tolist()

# Example output (384 floats):
# embed_query("What is RAG?") → [0.021, -0.043, 0.118, 0.002, ...] (384 numbers)

---
## 3. RAG Part 2 — Retrieval at Query Time <a id='3-rag-retrieval'></a>

At query time, the process is:

```
User question
  │
  ▼  embed_query(question) → 384-float vector
FAISS similarity_search(vector, k=5)
  │
  ▼  returns top 5 child chunks (smallest cosine distance)
parent_docs lookup (by parent_id in metadata)
  │
  ▼  returns the full parent section (2048 chars)
Inject into LLM prompt as context
```

The FAISS index lives in **RAM** (app.state.vectorstore). Loading it at startup takes ~0.5s. Every request after that is a pure in-memory vector search — no disk IO, no database round trip.

In [ ]:
# backend/agent/nodes/rag_worker.py — the RAG worker that runs during Phase 1

import json, re
from agent.state import AgentState
from config import settings

# Stop words: common English words that carry no semantic meaning.
# Filtering them out makes the relevance heuristic more accurate.
_STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from",
    "how", "in", "into", "is", "it", "of", "on", "or", "that", "the", "their",
    "this", "to", "what", "when", "where", "which", "why", "with", "you", "your",
}

async def rag_worker_node(state: AgentState, tools: list) -> AgentState:
    send = state["send"]
    await send({"type": "worker_status", "worker": "rag", "status": "Searching book…"})

    # tools are bound at request time (they reference the loaded FAISS index)
    tool_map = {t.name: t for t in tools}
    rag_chunks = []

    search_tool = tool_map.get("rag_search")
    if search_tool:
        # The tool runs: vectorstore.similarity_search(query, k=5)
        # Then for each result, it looks up the parent section by parent_id.
        result_json = search_tool.invoke({"query": state["user_message"], "k": settings.rag_top_k})
        rag_chunks = json.loads(result_json)
        # Each chunk: {text, chapter, page_number, section, parent_id}

    # Relevance heuristic — decides if the retrieval looks good
    relevance, notice = _assess_retrieval_relevance(state["user_message"], rag_chunks)

    return {**state, "rag_chunks": rag_chunks, "retrieval_relevance": relevance, "retrieval_notice": notice}


def _assess_retrieval_relevance(query, rag_chunks):
    """
    Cheap word-overlap heuristic to decide if the retrieved content looks relevant.

    Not ML-based — just checks if query terms appear in the retrieved text.
    Conservative: only flags "weak" when coverage genuinely looks thin.
    """
    if not rag_chunks:
        return ("weak", "I did not find a strong match for this question in the book alone.")

    query_terms = _meaningful_terms(query)
    if not query_terms:
        return ("strong", "")

    combined_text = "\n".join(chunk.get("text", "") for chunk in rag_chunks[:3]).lower()
    matched_terms = [term for term in query_terms if term in combined_text]
    coverage = len(matched_terms) / max(len(query_terms), 1)

    if len(rag_chunks) < 2 and coverage < 0.34:
        return ("weak", "The book retrieval looks only loosely related here.")
    if coverage < 0.2:
        return ("weak", "I found only a weak match in the book for this question.")

    return ("strong", "")


def _meaningful_terms(text):
    # Extract words ≥3 chars, filter stop words
    words = re.findall(r"[a-zA-Z][a-zA-Z0-9-]{2,}", text.lower())
    return [word for word in words if word not in _STOP_WORDS]

### Why is the relevance check a word-overlap heuristic and not another LLM call?

Calling an LLM to evaluate the quality of a retrieval would add latency and cost to every single query. A simple term-overlap check is good enough for this purpose — it catches cases where the query is about a vendor-specific product not mentioned in the book (e.g. "LangSmith"), and surfaces a notice to the user offering web search as an escalation. Precision does not need to be high here.

---
## 4. Agentic Orchestration — The 4-Phase Pipeline <a id='4-agent-pipeline'></a>

### What is an agent?

A traditional API call is: input → LLM → output. Done.

An **agent** can:
- Decide *what to do* based on the input (routing)
- Use **tools** — functions it can call (like searching a database or the web)
- Produce **structured outputs** (like JSON for a knowledge graph)
- Run multiple steps in a coordinated sequence

The word "agent" just means the system has some autonomy about how it handles a request — it's not a single fixed prompt.

### Why a custom pipeline instead of LangGraph?

LangGraph is a popular framework for building agent pipelines. We chose not to use it here because:
- This pipeline has a fixed structure (always 4 phases in order)
- Explicit `asyncio` code is easier to trace when debugging
- No added abstraction layer between you and what's actually happening

The design decision: **explicit > magical** for a system this size.

### The pipeline

```
Phase 0  — ROUTE
           Orchestrator reads the question, outputs: SIMPLE | MEMORY | SEARCH
           ● SIMPLE → skip everything, answer directly with Haiku (fast + cheap)
           ● MEMORY → use session history as context
           ● SEARCH → run the full pipeline below

Phase 1a — PARALLEL (asyncio.gather)
           rag_worker    → searches FAISS, returns book sections
           research_worker → (optional) DuckDuckGo web search

Phase 1b — SERIAL (depends on 1a output)
           graph_worker → reads RAG chunks + research,
                          decides NO_GRAPH | replace | update | new_chat
                          outputs JSON graph for the frontend canvas

Phase 2  — SYNTHESISE
           Orchestrator reads ALL context (chunks + research + graph)
           Streams response tokens back to browser via SSE

Phase 3  — ASYNC NODE ENRICHMENT (fire and forget)
           After "done" event sent, runs N workers in background
           Each worker enriches one graph node with description + book refs
           Sends node_detail SSE events as they complete
```

In [ ]:
# backend/agent/graph.py — the top-level pipeline runner

from agent.nodes.orchestrator_node import orchestrator_route, orchestrator_synthesise, quick_synthesise
from agent.pipeline_steps import (
    apply_graph_worker,
    maybe_expand_with_search_tool,
    maybe_start_node_enrichment,
    run_parallel_research_phase,
    run_search_phase,
)
from agent.state import AgentState


async def run_agent(state: AgentState, rag_tools, graph_tools, node_detail_tools) -> AgentState:
    """
    AgentState is a dict that accumulates data as it flows through each phase.
    Every node receives the whole state and returns an updated copy.
    """

    # Phase 0: Routing
    state = await orchestrator_route(state)
    # state["route"] is now "simple", "memory", or "search"

    if state["route"] == "simple":
        # Fast exit: Haiku answers directly, no RAG, no graph
        return await quick_synthesise(state)

    research_enabled = state.get("research_enabled", False)

    # Phase 1a: Parallel RAG + optional research
    if research_enabled:
        # asyncio.gather runs both workers simultaneously
        state = await run_parallel_research_phase(state, rag_tools)
        search_tool_wait_task = None
    else:
        # Run RAG only; set up a wait slot in case user clicks "Add web search"
        state, search_tool_wait_task = await run_search_phase(state, rag_tools)

    # Phase 1b: Graph worker (needs RAG chunks from 1a)
    state = await apply_graph_worker(state, graph_tools)

    # If user clicked "Add web search" while we were computing,
    # re-run graph worker with the wider evidence bundle before synthesising
    state = await maybe_expand_with_search_tool(state, graph_tools, search_tool_wait_task)

    # Phase 2: Synthesise + stream response
    state = await orchestrator_synthesise(state)

    # Phase 3: Non-blocking node enrichment (starts AFTER done event is sent)
    maybe_start_node_enrichment(state, node_detail_tools)
    # This does asyncio.create_task(...) — the task runs in the background.
    # The HTTP response is already delivered; this does not add any user-visible latency.

    return state

### What is AgentState?

`AgentState` is a Python `TypedDict` — essentially a dict with known keys and type annotations. It acts as the shared data container that all pipeline phases read from and write to.

Every node receives the full state and returns `{**state, "key": new_value}` — a copy with the relevant fields updated. This immutable pattern makes the data flow easy to trace: you can print `state` at any point and see exactly what every previous phase produced.

Key fields:
- `user_message` — what the user typed
- `history` — recent conversation turns
- `route` — set by Phase 0
- `rag_chunks` — set by Phase 1a (RAG worker)
- `research_context` — set by Phase 1a (research worker, if enabled)
- `graph_data` — set by Phase 1b (graph worker)
- `response_text` — set by Phase 2
- `send` — async callback that pushes SSE events to the browser

### Parallelism with asyncio.gather

Python's `asyncio` is single-threaded — only one coroutine runs at a time. But when a coroutine is *waiting* on IO (e.g. an API call), the event loop can switch to another coroutine.

The RAG search and web search are both IO-bound (API calls). Running them in parallel saves the full duration of the slower one:

```
Sequential:    RAG (0.3s) →→→ Research (1.5s) = 1.8s total
Parallel:      RAG (0.3s)
               Research (1.5s)  = 1.5s total (both finish by the slowest one)
```

In [ ]:
# Conceptual example of asyncio.gather (the pattern used in run_parallel_research_phase)

import asyncio

async def rag_task(state, tools):
    # simulate 0.3s for FAISS search
    await asyncio.sleep(0.3)
    return {**state, "rag_chunks": ["chunk A", "chunk B"]}

async def research_task(state):
    # simulate 1.5s for DuckDuckGo
    await asyncio.sleep(1.5)
    return {**state, "research_context": "web result X"}

async def run_parallel_demo(state):
    # asyncio.gather runs both coroutines concurrently.
    # It returns when BOTH are done — so total time = max(0.3, 1.5) = 1.5s.
    rag_state, research_state = await asyncio.gather(
        rag_task(state, tools=[]),
        research_task(state),
    )
    # Merge the results into a single state dict
    return {
        **state,
        "rag_chunks":       rag_state["rag_chunks"],
        "research_context": research_state["research_context"],
    }

---
## 5. Prompt Engineering — Every System Prompt Explained <a id='5-prompt-engineering'></a>

### What is a system prompt?

Every LLM call has two parts:
- **System prompt** — persistent instructions that define the model's role and behaviour
- **User messages** — the actual conversation history + current question

The system prompt is like a job description. It tells the model *who it is*, *what it should do*, and *what it should never do*.

We have three system prompts for three different jobs:

1. **Router** — classify the question (SIMPLE / MEMORY / SEARCH)
2. **Quick synthesis** — short factual answer (no RAG context)
3. **Full synthesis** — comprehensive answer grounded in book evidence

Plus a fourth system prompt for the graph worker (covered separately).

In [ ]:
# backend/agent/nodes/orchestrator_node.py — the router system prompt

_ROUTER_SYSTEM = """<role>
You are the router for an AI study assistant specialised in the book "AI Engineering" by Chip Huyen.
</role>

<task>
Classify the user's new turn into exactly one route token.
</task>

<output_contract>
Return EXACTLY one token and nothing else:
SIMPLE
MEMORY
SEARCH
</output_contract>

<decision_policy>
SIMPLE
- Short factual question answerable in 2-4 sentences from general AI / ML knowledge.
- Good examples: "what is X?", "what does X stand for?", "define X"
- Do NOT use SIMPLE for build, design, implementation, or architecture questions.

MEMORY
- The answer depends on earlier conversation context.
- Use MEMORY for references like "that earlier idea", "what did we decide".

SEARCH
- A fresh book search is needed.
- Always use SEARCH when the user asks to expand, add more nodes, or dig deeper.
- Always use SEARCH for build / design / implementation questions.
</decision_policy>

<guardrails>
- Be conservative. If the turn could reasonably need new evidence, choose SEARCH.
- Do not explain your choice.
- Do not output punctuation, JSON, or extra words.
</guardrails>"""

# Why XML-style tags?
# Anthropic's Claude models are fine-tuned to follow instructions in structured XML tags.
# <role>, <task>, <output_contract> are not special — they're just clear section markers.
# The model has learned that these delimit distinct instruction categories.

# Why specify <output_contract>?
# Without it, the model might output: "I'll classify this as SEARCH because the user
# is asking about architecture." We only want the single token "SEARCH".
# An explicit output contract reduces parsing errors dramatically.

In [ ]:
# backend/agent/nodes/orchestrator_node.py — the full synthesis system prompt

_SYNTHESIS_SYSTEM = """<role>
You are a study assistant for "AI Engineering" by Chip Huyen (O'Reilly).
Your audience is new to AI and systems work.
</role>

<core_task>
Write a concise beginner-friendly explanation grounded in the retrieved book evidence.
If graph context is provided, anchor the explanation to the exact node labels, sequence steps,
lanes, or groups when useful.
</core_task>

<style>
- For non-trivial explanations, use 3-5 short chunks.
- Each chunk: `Topic: locator` on one line, then 1-2 short sentences.
- Keep it to about 120-220 words total before follow-up suggestions.
- End with `If you want, I can:` and 2-3 short follow-up bullets.
- Cite inline as (Chapter N, p.X) only when it adds value.
</style>

<failure_avoidance>
- Do not write dense wall-of-text paragraphs.
- Do not invent graph positions or edge directions not supported by the provided graph context.
- Do not mention the graph unless it exists or the user asked about it.
</failure_avoidance>"""

# Key design choices:
#
# 1. Word count target (120-220)
#    Without this, the model tends to write 400+ word essays.
#    A specific range gives something to aim for.
#
# 2. "anchor to node labels" — graph-aware synthesis
#    The graph worker already built a diagram showing node A → B → C.
#    By asking synthesis to reference those exact labels, the written explanation
#    and the visual diagram reinforce each other.
#
# 3. <failure_avoidance> block
#    Listing what NOT to do is often more effective than positive instructions.
#    Models have a default tendency toward dense prose; explicitly naming it as
#    a failure mode suppresses it reliably.

### The graph worker prompt — structured JSON output

The graph worker's job is to decide what to do with the knowledge graph and output it as JSON. This is the most constrained prompt in the system — the model is told to output *only* a JSON object with a strict schema.

This is an example of **constrained generation**: the model acts as a JSON serialiser, not a conversational assistant.

In [ ]:
# backend/agent/nodes/graph_worker.py — excerpt of the graph worker system prompt

_GRAPH_SYSTEM = """<task>
Decide how to update the knowledge graph shown alongside the chat.
The graph must stay grounded in the retrieved book evidence, not in generic infra patterns.
</task>

<decision_policy>
Pick ONE action:

1. NO_GRAPH  — short follow-up, existing graph already covers the topic
2. "replace" — topic has shifted; build a fresh graph
3. "update"  — same topic; output only the NEW nodes and edges to merge in
4. "new_chat"— unrelated topic; suggest starting a fresh thread
</decision_policy>

<output_contract>
When outputting a graph, respond with ONLY a JSON object:
{
  "action": "replace" | "update" | "new_chat",
  "title": "<2-5 word title>",
  "nodes": [
    {
      "id": "<snake_id>",
      "label": "<MAX 3 words, MAX 20 chars>",
      "type": "<client|service|datastore|gateway|network|external|decision>",
      "technology": "<MAX 25 chars — e.g. 'Python / FastAPI', 'FAISS'>",
      "description": "<1 sentence>"
    }
  ],
  "edges": [
    {"source": "<id>", "target": "<id>", "label": "<2-4 word verb phrase>"}
  ]
}
</output_contract>"""

# Why "action" as an enum inside the JSON?
#   The same output format handles three different actions.
#   Parsing is always the same: load JSON, read action field, branch.
#   A simpler alternative (different formats per action) would require
#   detecting the action before parsing — fragile.
#
# Why MAX 3 words / MAX 20 chars on node labels?
#   D3 renders these labels inside SVG boxes. Long labels overflow.
#   Constraints in the prompt prevent UI breakage from over-verbose outputs.
#
# What happens if the model outputs invalid JSON?
#   The graph_worker code has a try/except around json.loads().
#   On failure: state["graph_data"] is left unchanged (existing graph kept).
#   The synthesis step still runs — the user gets an explanation, just no new graph.

### Complexity hints — user-controlled prompt injection

The user can choose a complexity level (low / prototype / production) from the UI. This choice is injected as a prefix into the graph worker's system prompt. The graph worker then adjusts how many nodes it generates.

In [ ]:
# backend/agent/nodes/graph_worker.py — complexity hint injection

_COMPLEXITY_HINTS = {
    "low":        "COMPLEXITY: low — 3-5 nodes max. Concept-diagram style. No groups, no networking.",
    "prototype":  "COMPLEXITY: prototype — 5-8 nodes. Basic architecture only. Skip observability.",
    "production": "COMPLEXITY: production — 8-15 nodes. Full AWS-style: all networking, monitoring lane.",
}

def _build_system_with_hint(complexity):
    hint = _COMPLEXITY_HINTS.get(complexity, "")
    if hint:
        return f"{hint}\n\n{_SYSTEM}"
    return _SYSTEM

# Pattern: the system prompt is *mostly* fixed, with a small dynamic injection
# for user preferences. This keeps the core instructions stable and easy to
# reason about, while allowing controlled variation.
#
# A common mistake: making the whole system prompt dynamic.
# This makes debugging harder because the model's behaviour changes unpredictably.

---
## 6. Streaming — How Tokens Reach the Browser <a id='6-streaming-sse'></a>

### Why streaming?

LLMs generate text one token at a time. Without streaming, the user would wait 5-10 seconds staring at a blank screen, then see the full answer appear at once. With streaming, tokens appear as they are generated — the first word shows up in ~0.5s.

### SSE vs WebSocket

- **WebSocket** — full duplex (both sides can send at any time). Complex to manage. Good for real-time chat or games.
- **SSE** (Server-Sent Events) — one-way: server pushes events to browser over a long-lived HTTP connection. Simpler. Perfect for LLM streaming.

The codebase has a hook named `useWebSocket.ts` — but it actually uses SSE underneath. Naming artifact from an earlier prototype.

### The queue-bridge pattern

The key problem: the agent pipeline (`run_agent`) and the SSE generator (which yields events to the browser) run in different coroutines. They need to communicate.

Solution: an `asyncio.Queue`. The agent calls `send(event)` to put events in the queue. The SSE generator drains the queue and yields each event as an SSE message.

In [ ]:
# backend/api/sse_handler.py — the queue-bridge pattern (simplified)

import asyncio

async def stream():
    # The queue bridges two concurrently running coroutines:
    #   Producer: run_agent() calls send() → puts events on queue
    #   Consumer: this generator drains the queue → yields SSE lines to browser
    queue = asyncio.Queue()

    async def send(event: dict) -> None:
        await queue.put(event)

    # Launch the agent as a background task.
    # create_task() schedules it but doesn't wait for it — execution continues immediately.
    agent_task = asyncio.create_task(
        run_agent(state, rag_tools, graph_tools, node_detail_tools)
    )

    # Drain the queue while the agent runs.
    while True:
        try:
            # wait_for(timeout=0.05) means: try to get an event from the queue.
            # If nothing arrives in 50ms, raise TimeoutError and re-check.
            event = await asyncio.wait_for(queue.get(), timeout=0.05)
            yield sse(event)   # forward to browser as SSE
        except asyncio.TimeoutError:
            if agent_task.done() and queue.empty():
                break  # agent is done and we've sent everything

# SSE wire format:
# Each event is two lines followed by a blank line:
#
#   data: {"type": "response_delta", "content": "Hello"}
#
#   data: {"type": "response_delta", "content": " world"}
#
#   data: {"type": "done"}
#
# The browser's EventSource API parses these lines and fires an event
# for each `data:` line.

### SSE event types in this system

| Event type | Payload | When sent |
|---|---|---|
| `worker_status` | `{worker, status}` | When each phase starts (shown in ThinkingIndicator) |
| `response_delta` | `{content}` | Each token from the LLM |
| `graph_data` | `{data: GraphData}` | Before synthesis starts (draws the graph) |
| `node_detail` | `{nodeId, description, refs}` | During Phase 3, one per node |
| `suggested_questions` | `{questions: string[]}` | After node click |
| `error` | `{content}` | On any failure |
| `done` | `{}` | When complete |

The frontend hook (`useAgentStream.ts`) handles each event type with a dedicated handler. When `done` arrives, the hook finalises state and re-enables the input.

---
## 7. Authentication — Supabase OTP + JWT <a id='7-authentication'></a>

### The auth flow

```
1. User enters email
2. Frontend calls POST /api/auth/request-otp
3. Backend calls Supabase: "send OTP to this email"
4. Supabase emails a 6-digit code
5. User enters code
6. Frontend calls POST /api/auth/verify-otp
7. Supabase verifies the code and returns a JWT (JSON Web Token)
8. Frontend stores JWT in memory (not localStorage for security)
9. All subsequent API calls include the JWT in the Authorization header
10. Backend verifies the JWT signature on every request
```

### What is a JWT?

A **JWT** (JSON Web Token) is a compact, self-contained token. It has three parts separated by dots:
- **Header** — algorithm used (HS256, RS256)
- **Payload** — the claims: user ID, email, expiry time
- **Signature** — cryptographic proof that the payload wasn't tampered with

The backend verifies the signature using a shared secret (for HS256) or a public key (for RS256). If the signature is valid and the token hasn't expired, the user is authenticated.

### Why OTP (one-time password) instead of email+password?

- No password to store (no bcrypt, no password reset flow)
- No password to be phished
- Supabase handles the email delivery and OTP lifecycle

Trade-off: requires access to email to log in. Acceptable for a study tool.

In [ ]:
# backend/adapters/supabase_auth_adapter.py — JWT verification (simplified)

import jwt   # PyJWT library
from fastapi import Depends, HTTPException
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer

security = HTTPBearer()

def verify_access_token(token: str) -> dict:
    """
    Verify a Supabase-issued JWT.
    Supabase issues HS256 tokens (symmetric secret) for auth.
    """
    try:
        payload = jwt.decode(
            token,
            key=settings.supabase_jwt_secret,   # shared secret from GCP Secret Manager
            algorithms=["HS256"],
            options={"verify_aud": False},       # Supabase tokens don't have `aud`
        )
        # payload now looks like:
        # {"sub": "user-uuid", "email": "user@example.com", "exp": 1712345678}
        return payload
    except jwt.ExpiredSignatureError:
        raise HTTPException(status_code=401, detail="Token expired")
    except jwt.InvalidTokenError:
        raise HTTPException(status_code=401, detail="Invalid token")


async def get_current_user(credentials: HTTPAuthorizationCredentials = Depends(security)):
    """
    FastAPI dependency — extracts and verifies the user from the Bearer token.
    Any endpoint that does `user=Depends(get_current_user)` requires a valid JWT.
    If the token is missing or invalid, FastAPI returns 401 before entering the handler.
    """
    payload = verify_access_token(credentials.credentials)
    return {"id": payload["sub"], "email": payload.get("email")}

# Usage in endpoints:
# @router.post("/chat")
# async def chat_endpoint(body: ChatRequest, user=Depends(get_current_user)):
#     user_id = user["id"]  # safe — JWT was verified before we got here

### Cloudflare Turnstile — bot protection

Turnstile is a CAPTCHA replacement. The frontend shows a Turnstile widget on the login screen. When the user passes, Turnstile gives the browser a one-time token. The backend verifies this token with Cloudflare before sending the OTP email.

Why? Without it, a bot could spam `POST /api/auth/request-otp` with thousands of email addresses, using up Supabase's email quota and potentially being used for email spam.

For local development, there are special test keys (site=`1x00000000000000000000AA`) that always pass without showing the widget.

---
## 8. Persistence — Threads, Messages, and Graphs <a id='8-persistence'></a>

### Data model

```
profiles
  └── user_id (from Supabase auth)
  └── email

chat_threads
  └── id (UUID)
  └── user_id (foreign key)
  └── title
  └── graph_data (JSONB — the full graph including view_state)
  └── updated_at

chat_messages
  └── id
  └── thread_id (foreign key)
  └── user_id
  └── role ("user" | "assistant")
  └── content
  └── created_at
```

All tables have **Row Level Security (RLS)** enabled in Supabase. This means database-level enforcement that `user_id` matches the authenticated user — even if there's a bug in the application code, one user cannot read another's threads.

### Thread limit enforcement

Users are capped at 5 threads. When a 6th thread is created, the oldest one is automatically deleted.

In [ ]:
# backend/storage/thread_store.py — create_thread with eviction policy (simplified)

from config import settings

def create_thread(user_id: str, title: str = "New chat") -> dict:
    """
    Create a new thread. If the user is at the limit, evict the oldest thread first.
    Eviction is "oldest by updated_at" — least recently used.
    """
    existing = list_threads(user_id)  # ordered by updated_at DESC

    if len(existing) >= settings.max_threads_per_user:
        # evict the oldest thread (last in the list)
        oldest = existing[-1]
        delete_thread(user_id, oldest["id"])

    # Insert new thread
    thread_id = str(uuid.uuid4())
    db.execute(
        "INSERT INTO chat_threads (id, user_id, title) VALUES (%s, %s, %s)",
        [thread_id, user_id, title]
    )
    return get_thread(user_id, thread_id)


def save_graph(user_id: str, thread_id: str, graph_data: dict) -> bool:
    """
    Persist a graph to the thread.
    Returns False (and does NOT save) if the serialised graph exceeds the size limit.
    This prevents pathological graphs from bloating the database.
    """
    import json
    serialised = json.dumps(graph_data)

    if len(serialised.encode()) > settings.max_graph_bytes:
        return False  # caller should surface this as a 413 or a warning

    db.execute(
        "UPDATE chat_threads SET graph_data = %s, updated_at = NOW() WHERE id = %s AND user_id = %s",
        [serialised, thread_id, user_id]
    )
    return True

### Graph layout persistence

When the user drags nodes around in the D3 graph, those positions are saved back to the database. This is the `view_state` field inside `graph_data`.

The frontend uses a **400ms debounce** — it waits 400ms after the last drag event before calling `PUT /api/threads/{id}/graph`. This batches rapid interactions into a single write.

```
User drags node A
User drags node A again (100ms later)
User drags node B (200ms later)
  ← 400ms of silence →
  API call with final positions  ✓
```

Without debouncing: hundreds of API calls per drag session. With debouncing: one call per "done dragging" event.

---
## 9. CI/CD — From Git Push to Live Production <a id='9-cicd'></a>

### What is CI/CD?

- **CI** (Continuous Integration) — automatically run tests whenever code is pushed
- **CD** (Continuous Deployment) — automatically deploy if tests pass

Without CI/CD, deploying requires: run tests → build Docker image → push to registry → update Cloud Run → check it works → push frontend to Vercel. Manually, this takes 15-30 minutes and is error-prone.

With CI/CD, a `git push` to `main` triggers the whole pipeline automatically.

### Pipeline structure

```
git push main
  │
  ├── Job 1: test-backend (pytest + real Postgres)
  └── Job 2: test-frontend (tsc + vite build)
      │
      ▼  (both must pass)
  Job 3: deploy-backend-candidate
         → docker buildx build → push to Artifact Registry
         → deploy to Cloud Run with --no-traffic (staging revision)
      │
      ▼
  Job 4: staging-eval
         → load test credentials from GCP Secret Manager
         → wait for /api/prepare to return 200
         → run eval suite against the staging revision
      │
      ▼  (only if eval passes)
  Job 5: promote-backend
         → shift 100% traffic to the staged revision
  Job 6: deploy-frontend → vercel build → vercel deploy --prod
```

In [ ]:
# .github/workflows/deploy.yml — annotated key sections

# ── Job 1: Backend tests ─────────────────────────────────────────────────────
# Why a postgres service block?
# The storage layer connects to Postgres at startup. Without a running DB,
# the tests fail with "connection refused".
# GitHub Actions can spin up a Docker container alongside the job runner:
"""
services:
  postgres:
    image: postgres:16
    env:
      POSTGRES_USER: ci
      POSTGRES_PASSWORD: ci
      POSTGRES_DB: ci
    ports:
      - 5432:5432
    options: --health-cmd pg_isready --health-interval 10s
"""

# ── Job 2: Frontend build ─────────────────────────────────────────────────────
# Why rm -f package-lock.json before npm install?
# Vite 6 uses rolldown, which has platform-specific native bindings.
# A lock file generated on macOS contains @rolldown/binding-darwin-arm64.
# The CI runner is Linux, which needs @rolldown/binding-linux-x64-gnu.
# npm ci would use the lock file and fail to resolve the correct binary.
# Fix: delete the lock file, npm install regenerates it for the current platform.
"""
- name: Install dependencies
  working-directory: frontend
  run: rm -f package-lock.json && npm install
"""

# ── Job 3: Docker build ────────────────────────────────────────────────────────
# Why docker buildx and not plain docker build?
# Plain `docker build` uses the default driver, which cannot export cache
# to a registry. buildx uses BuildKit, which supports --cache-to type=registry.
# Cache hit on the pip install layer (the heaviest step) cuts build time by 4-8x.
"""
- name: Set up Docker Buildx
  uses: docker/setup-buildx-action@v3

- name: Build and push Docker image
  run: |
    docker buildx build \\
      -f backend/Dockerfile \\
      --cache-from type=registry,ref=$IMAGE:buildcache \\
      --cache-to   type=registry,ref=$IMAGE:buildcache,mode=max \\
      --tag $IMAGE:$GITHUB_SHA \\
      --push \\
      .
"""
# --push: streams layers directly to Artifact Registry; no separate `docker push` step.
# --tag with $GITHUB_SHA: each commit gets a unique tag, so old images are traceable.

# ── Keyless GCP auth (Workload Identity Federation) ────────────────────────────
# Traditionally, CI would store a GCP service account key as a GitHub secret.
# Problem: long-lived keys are a security risk. If leaked, an attacker has GCP access.
# 
# WIF (Workload Identity Federation) is the keyless alternative:
# 1. GitHub issues a short-lived OIDC token for the running workflow
# 2. GCP exchanges that OIDC token for a short-lived GCP access token
# 3. No static key is ever stored
# 4. The GCP org policy actually blocks SA key creation entirely
"""
permissions:
  id-token: write   # required to request an OIDC token from GitHub

- name: Authenticate to Google Cloud
  uses: google-github-actions/auth@v2
  with:
    workload_identity_provider: ${{ secrets.GCP_WORKLOAD_IDENTITY_PROVIDER }}
    service_account: ${{ secrets.GCP_SERVICE_ACCOUNT }}
"""

In [ ]:
# .github/workflows/deploy.yml — staging eval + traffic promotion

# ── Job 3 deploys with --no-traffic ────────────────────────────────────────────
# Cloud Run allows deploying a new revision without routing any traffic to it.
# The revision gets a tag (staging-<run_id>) and a unique URL.
# We test against THAT URL before any real user sees the new code.
"""
- id: deploy
  uses: google-github-actions/deploy-cloudrun@v2
  with:
    service: agent-backend
    region: europe-west2
    image: ${{ env.IMAGE }}:${{ github.sha }}
    flags: --no-traffic --tag=${{ env.CANDIDATE_TAG }}   # ← zero traffic, tagged URL
"""

# ── Job 4: staging eval ─────────────────────────────────────────────────────────
# Waits for /api/prepare to return 200 (FAISS loaded, DB connected).
# Then runs run_staging_eval.py — a script that fires real queries and checks
# the responses meet quality thresholds.
"""
- name: Wait for backend candidate readiness
  run: |
    for attempt in $(seq 1 30); do
      status="$(curl -sS -w '%{http_code}' "$CANDIDATE_URL/api/prepare" || true)"
      if [ "$status" = "200" ]; then exit 0; fi
      sleep 10
    done
    exit 1   # never became ready after 5 minutes
"""

# ── Job 5: promote only after eval passes ──────────────────────────────────────
# If the eval fails, Job 5 is skipped. Production traffic stays on the old revision.
# No rollback needed — we never pointed traffic at the broken revision.
"""
- name: Shift production traffic to the staged revision
  run: |
    gcloud run services update-traffic agent-backend \\
      --region europe-west2 \\
      --to-tags $CANDIDATE_TAG=100
"""

---
## 10. Infrastructure — Cloud Run, Vercel, and Cold Starts <a id='10-infrastructure'></a>

### Cloud Run — cost-first backend

Cloud Run is a serverless container platform. You give it a Docker image; it handles everything else (scaling, HTTPS, load balancing). You pay only for CPU+RAM used while a request is being handled.

Key setting: `min-instances = 0`
- When no one is using the app, Cloud Run scales to zero containers
- Cost drops to ~$0/month when idle
- Trade-off: **cold start** — the first request after a period of inactivity waits for a container to spin up (~3-8 seconds)

### The cold-start UX pattern

Rather than hiding the latency or hoping the user doesn't notice, the frontend makes the cold start explicit:

```
Page loads
  → Send button is DISABLED (state: unknown)
  → "Prepare" button is shown

User clicks Prepare
  → GET /api/prepare (this HTTP call wakes the Cloud Run container)
  → While waiting: state = "preparing", spinner shown
  → /api/prepare returns 200 when FAISS is loaded and DB is connected
  → state = "ready", Send button ENABLED

10-12 minutes later (freshness window)
  → state resets to "unknown" (container might have gone cold again)
```

Why is this better than an automatic ping?
- Automatic pings (polling every N seconds) costs money even when no one is using the app
- The explicit "Prepare" teaches the user about the system's architecture (transparency)
- Users who don't want to wait can upgrade to a paid plan with `min-instances=1`

In [ ]:
# backend/api/health_route.py — the /api/prepare endpoint
# This is the startup probe that the frontend calls and that CI polls during staging eval.

from fastapi import APIRouter, Request
from fastapi.responses import JSONResponse

router = APIRouter()

@router.get("/api/prepare")
async def prepare(request: Request):
    """
    Readiness check. Returns 200 only when:
    - FAISS index is loaded into app.state.vectorstore
    - parent_docs are loaded into app.state.parent_docs
    - DB connection is healthy

    Cloud Run startup probe also hits this endpoint.
    Cloud Run will NOT route production traffic to a revision
    until this endpoint returns 200.
    """
    vectorstore = getattr(request.app.state, "vectorstore", None)
    parent_docs = getattr(request.app.state, "parent_docs", None)

    if not vectorstore or not parent_docs:
        return JSONResponse(status_code=503, content={"status": "loading"})

    return {"status": "ok"}

# Why 503 instead of 200 with a loading flag?
# The CI readiness poll checks: if status != 200, sleep and retry.
# A non-200 status code is the standard HTTP signal for "not ready yet".

### Vercel — frontend deployment

Vercel is optimised for frontend frameworks (React, Next.js, etc.).

Key facts:
- The React app is a **static site** — built to HTML + JS + CSS, served from a CDN
- No server-side rendering in this project; all interactivity is client-side
- Environment variables (`VITE_API_URL`, `VITE_SUPABASE_URL`, etc.) are baked into the build at compile time
- `vercel build --prod` builds the app using Vercel's build system; `vercel deploy --prebuilt` uploads the output

### Secret management

| Secret | Storage | Accessed by |
|---|---|---|
| Anthropic API key | GCP Secret Manager | Cloud Run (mounted as env var) |
| Supabase DB URL | GCP Secret Manager | Cloud Run |
| Supabase anon key | Vercel env vars | Frontend build |
| Turnstile site key | Vercel env vars | Frontend build |
| GCP WIF provider | GitHub secrets | CI/CD only |

Rule: secrets that backend code needs live in GCP Secret Manager. Secrets that frontend code needs are Vercel env vars (these become public in the browser bundle — only use non-sensitive public keys here).

---
## 11. Key Patterns Worth Remembering <a id='11-key-patterns'></a>

These are design decisions that recur across the codebase. Understanding the *why* behind each one will help you make similar decisions in your own projects.

---

### Pattern 1: Tools are bound at request time, not at startup

The RAG search tool needs access to the FAISS index, which lives in `app.state`. The tool is created fresh for each request and bound to the loaded vectorstore.

```python
# In sse_handler.py — per-request binding
rag_search_tool  = make_rag_search_tool(request.app.state.vectorstore, request.app.state.parent_docs)
get_section_tool = make_get_section_tool(request.app.state.parent_docs)
```

Why not create one global tool at startup? The tool would hold a reference to `app.state`, which doesn't exist at import time. Per-request creation is clean and explicit.

---

### Pattern 2: State flows forward through the pipeline — never backward

Each phase receives the full state and returns an updated copy. No phase modifies state in place. This makes it easy to log state at any point for debugging and avoids hidden dependencies between phases.

```python
# Immutable state update pattern
return {**state, "rag_chunks": new_chunks, "retrieval_relevance": relevance}
# {**state} copies all existing keys; the rest override specific keys.
```

---

### Pattern 3: Fire-and-forget for non-critical work

Node enrichment (Phase 3) runs after the `done` event. The user sees the full response immediately. The graph node details appear seconds later as the background workers complete.

```python
# maybe_start_node_enrichment calls:
asyncio.create_task(enrich_all_nodes(state, tools))
# create_task schedules the coroutine but does not await it.
# The function returns immediately; the task runs in the background.
```

When to use this: work that improves the UX but is not required for the primary response.
When NOT to use this: work that must complete before a response is correct (e.g. database writes that need to be durable).

---

### Pattern 4: Debounce before persistence

User interactions (dragging graph nodes) can fire dozens of events per second. Persisting each one would flood the database.

```typescript
// Frontend: 400ms debounce before calling PUT /api/threads/{id}/graph
const debouncedSave = useMemo(
  () => debounce((graphData) => updateThreadGraph(session, threadId, graphData), 400),
  [session, threadId]
);
```

Rule of thumb: any user action that fires continuously (scroll, drag, resize, keypress) should be debounced or throttled before triggering side effects.

---

### Pattern 5: Security gates before opening the stream

The SSE stream, once opened, is hard to interrupt cleanly. Security checks (size limits, rate limits, prompt injection) run synchronously *before* the stream is opened.

```python
@router.post("/chat")
async def chat_endpoint(body: ChatRequest, ...):
    # All of these checks run BEFORE return streaming_response(stream())
    if byte_len(content) > settings.max_message_bytes:
        return error_stream("Message too large")
    if not check_prompt_injection(content):
        return error_stream("Message blocked")
    if not check_rate_limit(user_id):
        return error_stream("Rate limited")
    # Only reach here if all checks pass
    return streaming_response(stream())
```

---

### Pattern 6: Configuration via pydantic-settings

All configuration lives in `config.py` using `pydantic-settings`. Values are read from environment variables, with validation and defaults. No hardcoded values in the business logic.

```python
# config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    anthropic_api_key: str
    max_threads_per_user: int = 5
    rag_top_k: int = 5
    agent_timeout_s: float = 90.0
    orchestrator_model: str = "claude-sonnet-4-6"
    worker_model: str = "claude-haiku-4-5-20251001"

    class Config:
        env_file = ".env"

settings = Settings()
```

In tests, you set dummy env vars before importing. In production, real values come from GCP Secret Manager mounted as env vars. The business logic never changes.

---

### Pattern 7: Orchestrator uses a powerful model; workers use a fast/cheap model

| Role | Model | Why |
|---|---|---|
| Router | claude-sonnet-4-6 | Needs nuanced judgement on question type |
| Graph worker | claude-sonnet-4-6 | Needs structured JSON output + architectural reasoning |
| Synthesis | claude-sonnet-4-6 | Needs high-quality writing |
| Quick synthesis | claude-haiku-4-5 | Simple factual Q&A — Haiku is fast + cheap enough |
| Node detail | claude-haiku-4-5 | Runs N times in parallel — cost matters |

Rule: use the smallest model that meets the quality bar for each job. The router classifies into 3 buckets — that doesn't require Sonnet, but mis-routing is expensive, so we keep it on Sonnet anyway.

---

## What to study next

If this notebook made sense, here are natural next topics:

- **Embeddings deeper dive** — how cosine similarity works, why dimensionality matters, what all-MiniLM-L6-v2 is trained on
- **LangGraph** — the framework we deliberately didn't use; understanding it will help you see the trade-offs of explicit vs graph-based pipelines
- **Token budgeting** — how context windows work, why chunk size matters for quality, how to measure retrieval precision
- **Postgres + pgvector** — an alternative to FAISS where vectors live in the same database as your other data
- **Terraform** — the infra in `infra/terraform/gcp/` that provisions Cloud Run, Secret Manager, and Artifact Registry
- **OpenTelemetry** — structured observability for distributed systems; the backend already logs per-request latency and status